# 03. Kalendarz: daty w `<chrono>` (C++20)

## Cele

Po tym notebooku potrafisz:

* zapisać datę jako `year_month_day` i zamienić ją na `sys_days` (i z powrotem),
* sprawdzić poprawność daty metodą `ok()` i wyjaśnić, dlaczego biblioteka w ogóle pozwala na niepoprawne daty,
* dodawać miesiące i lata do daty — i wiedzieć, co się dzieje na końcu miesiąca,
* liczyć dni między datami i wyznaczać dzień tygodnia,
* używać `last` („ostatni dzień miesiąca”, „ostatni piątek”) i `hh_mm_ss`.

Wszystko w tym notebooku działa na GCC 11 i Clang 14. Dat nie wypisujemy przez `operator<<`, bo te kompilatory
go jeszcze nie mają — każdy przykład ma małą funkcję `print`, która wypisuje pola daty jako liczby.

## Data jako pola i data jako licznik

Datę można przechowywać na dwa sposoby
([Hinnant, dokumentacja biblioteki *date*](https://howardhinnant.github.io/date/date.html)):

* **pola** — rok, miesiąc, dzień: `std::chrono::year_month_day`. Wygodne do czytania i do „dodaj miesiąc”.
* **licznik** — liczba dni od epoki: `std::chrono::sys_days`, czyli `time_point<system_clock, days>`.
  Wygodne do „dodaj 10 dni” i „ile dni minęło”.

Między nimi jest tania konwersja w obie strony, a `sys_days` łączy kalendarz z resztą `<chrono>`:
to zwykły punkt w czasie `system_clock` o dokładności jednego dnia.

Datę składa się operatorem `/` z wartości typów `year`, `month`, `day` (literały `2026y`, `9d`; stałe `November`):

```cpp
#include <chrono>
#include <iostream>

void print(std::chrono::year_month_day d) {
    std::cout << int(d.year()) << '-' << unsigned(d.month()) << '-' << unsigned(d.day()) << '\n';
}

int main() {
    using namespace std::chrono;
    year_month_day lab = 2026y / November / 9d;   // także: November / 9d / 2026y albo 9d / November / 2026y
    print(lab);

    sys_days serial = lab;                        // pola -> licznik dni
    std::cout << "dni od 1970-01-01: " << serial.time_since_epoch().count() << '\n';

    year_month_day back = serial + days{1};       // licznik + dni -> z powrotem na pola
    print(back);
}
```

Jest też `local_days` — ta sama liczba dni, ale „w lokalnej, jeszcze nieokreślonej strefie czasowej”
([cppreference: `local_t`](https://en.cppreference.com/cpp/chrono/local_t)). Przyda się w notebooku 05.

## Dodawanie dni: tylko przez `sys_days`

`year_month_day` obsługuje arytmetykę **miesięcy i lat**, ale **nie dni** — standard mówi to wprost:
*„year_month_day supports years- and months-oriented arithmetic, but not days-oriented arithmetic.
For the latter, there is a conversion to sys_days”*
([\[time.cal.ymd.overview\]](https://eel.is/c++draft/time.cal.ymd.overview#1)).
Dodanie dni do pól wymagałoby za każdym razem przeliczania przez licznik — biblioteka nie ukrywa tego kosztu
([FAQ biblioteki *date*](https://github.com/HowardHinnant/date/wiki/FAQ#why-cant-i-do-day-arithmetic-on-a-year_month_day)):

```cpp
#include <chrono>

int main() {
    using namespace std::chrono;
    year_month_day eve = 2024y / December / 31d;
    auto next = eve + days{1};  // błąd: brak operator+(year_month_day, days)
    (void)next;
}
```

Poprawnie: zamień na `sys_days`, dodaj, zamień z powrotem. Tak samo liczy się różnicę dni między datami:

```cpp
#include <chrono>
#include <iostream>

void print(std::chrono::year_month_day d) {
    std::cout << int(d.year()) << '-' << unsigned(d.month()) << '-' << unsigned(d.day()) << '\n';
}

int main() {
    using namespace std::chrono;
    year_month_day eve = 2024y / December / 31d;
    year_month_day next = sys_days{eve} + days{1};
    print(next);  // 2025-1-1: przejście przez koniec roku

    days until_lab = sys_days{2026y / November / 9d} - sys_days{2026y / October / 21d};
    std::cout << "do laboratorium: " << until_lab.count() << " dni\n";
}
```

## Niepoprawne daty i `ok()`

`year_month_day` może przechowywać datę, której nie ma w kalendarzu, np. 30 lutego. To celowa decyzja:
kompilator wyłapuje błędy typów, a to, co zostaje, sprawdza `ok()`
([cppreference: `ok`](https://en.cppreference.com/cpp/chrono/year_month_day/ok#Example),
[\[time.cal.ymd.members\]](https://eel.is/c++draft/time.cal.ymd.members#23)).
Czasem niepoprawna data jest błędem, a czasem wygodnym stanem pośrednim
([*date* wiki: „When is it ok to be !ok()?”](https://github.com/HowardHinnant/date/wiki/Examples-and-Recipes#not_ok_is_ok)).

```cpp
#include <chrono>
#include <iostream>

int main() {
    using namespace std::chrono;
    year_month_day a = 2026y / February / 30d;
    year_month_day b = 2026y / February / 28d;
    std::cout << "2026-02-30 ok? " << a.ok() << '\n';
    std::cout << "2026-02-28 ok? " << b.ok() << '\n';
}
```

## Dodawanie miesięcy i lat: pułapka końca miesiąca

`data + months{n}` zmienia rok i miesiąc, a **dzień zostawia bez zmian**
([\[time.cal.ymd.nonmembers\]](https://eel.is/c++draft/time.cal.ymd.nonmembers#4),
[cppreference, Notes](https://en.cppreference.com/cpp/chrono/year_month_day/operator_arith_2#Notes)).
Dla dni 1–28 wynik zawsze jest poprawny. Dla 29–31 — nie zawsze:

```cpp
#include <chrono>
#include <iostream>

void print(std::chrono::year_month_day d) {
    std::cout << int(d.year()) << '-' << unsigned(d.month()) << '-' << unsigned(d.day())
              << (d.ok() ? "" : "  <- !ok()") << '\n';
}

int main() {
    using namespace std::chrono;
    print(2025y / March / 31d + months{1});   // 31 kwietnia?
    print(2025y / March / 15d + months{1});   // bez problemu
}
```

Z latami jest tak samo — dokładnie ten błąd popełnił Windows Azure 29 lutego 2012, licząc datę ważności
certyfikatu jako „ten sam dzień za rok”
([Microsoft, 2012](https://azure.microsoft.com/en-us/blog/summary-of-windows-azure-service-disruption-on-feb-29th-2012/)):

```cpp
#include <chrono>
#include <iostream>

int main() {
    using namespace std::chrono;
    year_month_day issued = 2012y / February / 29d;
    year_month_day valid_to = issued + years{1};
    std::cout << "2013-02-29 ok? " << valid_to.ok() << '\n';
}
```

Co zrobić z niepoprawnym wynikiem — zależy od tego, czego chce biznes. Jedna z możliwości to przejście przez `sys_days`:
dla poprawnego roku i miesiąca konwersja „przelewa” nadmiarowe dni na następny miesiąc
([cppreference: `operator sys_days`](https://en.cppreference.com/cpp/chrono/year_month_day/operator_days#Notes)):

```cpp
#include <chrono>
#include <iostream>

void print(std::chrono::year_month_day d) {
    std::cout << int(d.year()) << '-' << unsigned(d.month()) << '-' << unsigned(d.day()) << '\n';
}

int main() {
    using namespace std::chrono;
    year_month_day bad = 2025y / June / 31d;
    year_month_day normalized = sys_days{bad};
    print(normalized);  // 31 czerwca "to" 1 lipca
}
```

Czy „1 lipca” to dobra odpowiedź na „miesiąc po 31 maja”? Nie zawsze — warto o tym pomyśleć przed ćwiczeniem.

## Dzień tygodnia, „n-ty” i „ostatni”

`weekday` da się zbudować z `sys_days` (dzień tygodnia zależy od numeru dnia, nie od pól)
([cppreference: `weekday`](https://en.cppreference.com/cpp/chrono/weekday/weekday)).
`c_encoding()` zwraca numer 0–6, gdzie 0 to niedziela.

Do opisów typu „drugi wtorek” i „ostatni piątek” służą `weekday_indexed` (`Tuesday[2]`) i znacznik `last`
([cppreference: `weekday_indexed`](https://en.cppreference.com/cpp/chrono/weekday_indexed),
[`last_spec`](https://en.cppreference.com/cpp/chrono/last_spec)). `last` działa też dla dni miesiąca:
`2028y / February / last` to „ostatni dzień lutego 2028”, a typ `year_month_day_last` sam pilnuje,
ile dni ma miesiąc ([cppreference](https://en.cppreference.com/cpp/chrono/year_month_day_last)).

```cpp
#include <chrono>
#include <iostream>

void print(std::chrono::year_month_day d) {
    std::cout << int(d.year()) << '-' << unsigned(d.month()) << '-' << unsigned(d.day()) << '\n';
}

int main() {
    using namespace std::chrono;
    const char* names[] = {"nd", "pn", "wt", "śr", "cz", "pt", "sb"};

    weekday wd{sys_days{2026y / November / 9d}};
    std::cout << "9 listopada 2026 to: " << names[wd.c_encoding()] << '\n';

    year_month_day patch_tuesday = sys_days{2026y / December / Tuesday[2]};  // drugi wtorek grudnia
    print(patch_tuesday);

    year_month_day last_feb = 2028y / February / last;  // ostatni dzień lutego
    print(last_feb);

    year_month_day last_friday = sys_days{2026y / July / Friday[last]};  // ostatni piątek lipca
    print(last_friday);
}
```

Dni tygodnia mają własną arytmetykę „modulo 7”: `Saturday + days{2} == Monday`, a różnica dwóch `weekday` to liczba dni
od jednego do drugiego, zawsze w przód (0–6).

## Godzina w ciągu dnia: `hh_mm_ss`

`hh_mm_ss` rozkłada odcinek czasu na godziny, minuty, sekundy i ułamek sekundy
([cppreference: `hh_mm_ss`](https://en.cppreference.com/cpp/chrono/hh_mm_ss)). Razem z `floor<days>` daje
rozbicie punktu w czasie na datę i godzinę
([*date* wiki: components from a time_point](https://github.com/HowardHinnant/date/wiki/Examples-and-Recipes#components_to_time_point)):

```cpp
#include <chrono>
#include <iostream>

int main() {
    using namespace std::chrono;
    sys_time<milliseconds> tp = sys_days{2026y / November / 9d} + 9h + 45min + 30s + 250ms;

    sys_days day = floor<days>(tp);          // data
    hh_mm_ss<milliseconds> time{tp - day};   // reszta: godzina w ciągu dnia
    year_month_day ymd = day;

    std::cout << int(ymd.year()) << '-' << unsigned(ymd.month()) << '-' << unsigned(ymd.day()) << ' '
              << time.hours().count() << ':' << time.minutes().count() << ':'
              << time.seconds().count() << '.' << time.subseconds().count() << '\n';
}
```

Uwaga na `floor`, a nie `duration_cast`: dla punktów sprzed 1970 roku `duration_cast` zaokrągliłby w stronę zera,
czyli **w przyszłość** — na następny dzień.

## Ciekawostka: lata przestępne i `2100y.is_leap()`

Rok jest przestępny, gdy dzieli się przez 4, z wyjątkiem lat podzielnych przez 100, które nie dzielą się przez 400
([\[time.cal.year.members\]](https://eel.is/c++draft/time.cal.year.members#16)). Rok 2100 przestępny nie będzie.
Ale sprawdzenie tego „wprost” się nie kompiluje:

```cpp
#include <chrono>

int main() {
    using namespace std::chrono;
    bool leap = 2100y.is_leap();  // błąd leksera!
    (void)leap;
}
```

Lekser czyta znaki zachłannie (*maximal munch*) i bierze `2100y.is_leap` za **jeden** token liczbowy
([\[lex.ppnumber\]](https://eel.is/c++draft/lex.ppnumber), [\[lex.pptoken\]/5](https://eel.is/c++draft/lex.pptoken#5.4)).
Wystarczą nawiasy albo spacja:

```cpp
#include <chrono>
#include <iostream>

int main() {
    using namespace std::chrono;
    std::cout << "2100: " << (2100y).is_leap() << ", 2000: " << (2000y).is_leap() << '\n';
}
```

## Ćwiczenie 3

*Treść ćwiczenia — w przygotowaniu.*

## Linki i materiały

* [cppreference: `year_month_day`](https://en.cppreference.com/cpp/chrono/year_month_day),
  [`weekday`](https://en.cppreference.com/cpp/chrono/weekday/weekday),
  [`last_spec`](https://en.cppreference.com/cpp/chrono/last_spec),
  [`hh_mm_ss`](https://en.cppreference.com/cpp/chrono/hh_mm_ss)
* Standard C++ (szkic roboczy): [\[time.cal.ymd\]](https://eel.is/c++draft/time.cal.ymd.overview#1)
* Howard Hinnant, biblioteka *date* (z niej powstał kalendarz C++20):
  [dokumentacja](https://howardhinnant.github.io/date/date.html),
  [FAQ](https://github.com/HowardHinnant/date/wiki/FAQ#why-cant-i-do-day-arithmetic-on-a-year_month_day),
  [przepisy i przykłady](https://github.com/HowardHinnant/date/wiki/Examples-and-Recipes#not_ok_is_ok)
* Bartłomiej Filipek, [„Exploring C++20 std::chrono — Calendar Types”](https://www.cppstories.com/2025/chrono-calendar-types/) —
  przegląd typów kalendarza

---

[Strona główna](../_start.ipynb)